# Day 2 | Notebook 5: RAG Evaluation
**RedisVL Focus:** `HFTextVectorizer` (for scoring), faithfulness guard, golden dataset testing

---
## 📌 Learning Objectives
1. Implement RAGAS-style evaluation metrics from scratch
2. Build and run a golden dataset test suite
3. Detect hallucinations automatically using faithfulness scoring
4. Compare two RAG configurations and detect regressions
---

In [3]:
# ─── INSTRUCTOR SETTINGS ─────────────────────────────────────────
SHOW_INSIGHTS = False
REDIS_URL = "redis://redis-vector-db:6379"
# ─────────────────────────────────────────────────────────────────
import numpy as np
from redisvl.utils.vectorize import HFTextVectorizer
from redisvl.index import SearchIndex
from redisvl.schema import IndexSchema
from redisvl.query import VectorQuery
from redisvl.extensions.llmcache import SemanticCache

vectorizer = HFTextVectorizer("sentence-transformers/all-MiniLM-L6-v2")
print("✅ Ready")

/tmp/ipykernel_176/1688929727.py:10: DeprecationWarning: Importing from redisvl.extensions.llmcache is deprecated. Please import from redisvl.extensions.cache.llm instead.
  from redisvl.extensions.llmcache import SemanticCache


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Ready


## 📌 Concept: RAGAS Evaluation Framework

| Metric | Formula | Meaning |
|---|---|---|
| **Faithfulness** | `claims_in_context / total_claims` | Is the answer grounded? |
| **Answer Relevancy** | `cosine(embed(Q), embed(A))` | Is the answer on-topic? |
| **Context Recall** | `ground_truth_facts_found / total_gt_facts` | Did we retrieve enough? |
| **Context Precision** | `useful_docs / retrieved_docs` | Did we retrieve noise? |

**The Golden Dataset**: A curated set of `(question, ground_truth, ideal_context_doc)` triples.
You build this manually from your domain. Then you test your RAG against it and track scores over time.


In [2]:
# Cell 1: Golden dataset — 8 question/answer/context triples
GOLDEN = [
    {
        "question": "What GPU does the Aether X1 have?",
        "ground_truth": "The Aether X1 has an NVIDIA RTX 4090 GPU with liquid cooling.",
        "ideal_doc_id": "doc_aether_x1"
    },
    {
        "question": "How long is the Sony XM5 battery life?",
        "ground_truth": "The Sony WH-1000XM5 battery lasts 30 hours with ANC on and 40 hours without.",
        "ideal_doc_id": "doc_sony_xm5"
    },
    {
        "question": "What is the price of the Nebula Tab Pro?",
        "ground_truth": "The Nebula Tab Pro starts at $899.",
        "ideal_doc_id": "doc_nebula_tab"
    },
    {
        "question": "How long does standard shipping take?",
        "ground_truth": "Standard shipping takes 3-5 business days.",
        "ideal_doc_id": "doc_shipping"
    },
    {
        "question": "Can I return a product after 30 days?",
        "ground_truth": "No. Products can only be returned within 30 days of purchase.",
        "ideal_doc_id": "doc_returns"
    },
    {
        "question": "What does the Aether warranty cover?",
        "ground_truth": "Aether warranty covers manufacturing defects for 3 years.",
        "ideal_doc_id": "doc_warranty"
    },
    {
        "question": "What storage options does the Nebula Tab Pro have?",
        "ground_truth": "The Nebula Tab Pro comes in 256GB, 512GB, and 1TB storage options.",
        "ideal_doc_id": "doc_nebula_tab"
    },
    {
        "question": "How much does express delivery cost?",
        "ground_truth": "Express delivery costs $15 for 1-2 day delivery.",
        "ideal_doc_id": "doc_shipping"
    },
]
print(f"✅ Golden dataset: {len(GOLDEN)} Q&A pairs")

✅ Golden dataset: 8 Q&A pairs


In [3]:
# Cell 2: Implement faithfulness score from scratch
def faithfulness_score(answer: str, context: str) -> float:
    """
    Faithfulness = fraction of meaningful answer words found in context.
    A simple but effective proxy for hallucination detection.
    """
    stop_words = {"the", "a", "an", "is", "are", "was", "it", "in", "of",
                  "and", "or", "to", "for", "with", "on", "this", "that"}
    context_words = set(context.lower().split()) - stop_words
    answer_words = [w for w in answer.lower().split()
                    if len(w) > 3 and w not in stop_words]
    if not answer_words:
        return 0.0
    hits = sum(1 for w in answer_words if w in context_words)
    return round(hits / len(answer_words), 4)

# Test: grounded vs hallucinated
context = "The Aether X1 has an RTX 4090 GPU and costs $2,499."
grounded = "The Aether X1 GPU is RTX 4090 with a price of 2499."
hallucinated = "The Aether X1 uses a solar-powered quantum processor and costs $999."

print(f"Grounded answer faithfulness    : {faithfulness_score(grounded, context):.3f}")
print(f"Hallucinated answer faithfulness: {faithfulness_score(hallucinated, context):.3f}")

if SHOW_INSIGHTS:
    print("\n💡 INSIGHT: 'solar-powered', 'quantum', '$999' are not in the context.")
    print("   Low faithfulness = hallucination. We block answers below threshold 0.4.")

Grounded answer faithfulness    : 0.500
Hallucinated answer faithfulness: 0.286


In [4]:
# Cell 3: Implement answer relevancy score using HFTextVectorizer + cosine similarity
def cosine_similarity(vec_a: list, vec_b: list) -> float:
    a, b = np.array(vec_a), np.array(vec_b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

def answer_relevancy_score(question: str, answer: str) -> float:
    """Higher = answer is more on-topic to the question."""
    q_vec = vectorizer.embed(question, as_buffer=False)
    a_vec = vectorizer.embed(answer, as_buffer=False)
    return round(cosine_similarity(q_vec, a_vec), 4)

q = "What GPU does the Aether X1 have?"
on_topic   = "The Aether X1 has an NVIDIA RTX 4090 GPU."
off_topic  = "The weather in Bangkok is hot and humid year-round."

print(f"On-topic answer relevancy : {answer_relevancy_score(q, on_topic):.4f}")
print(f"Off-topic answer relevancy: {answer_relevancy_score(q, off_topic):.4f}")

On-topic answer relevancy : 0.8626
Off-topic answer relevancy: -0.0930


In [5]:
# Cell 4: Implement context recall
def context_recall_score(ground_truth: str, retrieved_context: str) -> float:
    """
    Context Recall = fraction of ground_truth key facts found in retrieved context.
    """
    stop_words = {"the", "a", "an", "is", "are", "was", "it", "in", "of",
                  "and", "or", "to", "for", "with", "on"}
    gt_words = [w for w in ground_truth.lower().split()
                if len(w) > 3 and w not in stop_words]
    ctx_words = set(retrieved_context.lower().split())
    if not gt_words:
        return 0.0
    hits = sum(1 for w in gt_words if w in ctx_words)
    return round(hits / len(gt_words), 4)

gt = "The Sony WH-1000XM5 battery lasts 30 hours with ANC on."
perfect_ctx = "Battery life is 30 hours with ANC on, 40 hours without. Supports LDAC."
partial_ctx = "The Sony headphones are very popular for their noise cancellation."

print(f"Context recall (perfect): {context_recall_score(gt, perfect_ctx):.3f}")
print(f"Context recall (partial): {context_recall_score(gt, partial_ctx):.3f}")

Context recall (perfect): 0.400
Context recall (partial): 0.200


In [6]:
# Cell 5: Rebuild the PDR index and run full evaluation
# (Reuse knowledge base from NB04 — ensure NB04 was run first,
#  OR re-ingest the same KNOWLEDGE_BASE here)

KNOWLEDGE_BASE = {
    "doc_aether_x1": "The Aether X1 Flagship laptop features an NVIDIA RTX 4090 GPU with a proprietary liquid-cooling system. It has a 4K OLED 120Hz ProMotion display, 64GB DDR5 RAM, and 2TB NVMe SSD. The price is $2,499 with a 3-year global warranty.",
    "doc_sony_xm5": "The Sony WH-1000XM5 headphones feature industry-leading active noise cancellation with dual-chip processing. Battery life is 30 hours with ANC on, 40 hours without. Supports LDAC, AAC, and SBC codecs. Price is $349.",
    "doc_nebula_tab": "The Nebula Tab Pro is a 12-inch OLED tablet with an Apple M3 chip. It supports the Aether Pencil 2nd generation. Storage options: 256GB, 512GB, 1TB. Starting price $899.",
    "doc_warranty": "All Aether products come with a 3-year global warranty covering manufacturing defects. Accidental damage is covered under AetherCare+ for $99/year. Claims via any Aether Store or online portal.",
    "doc_shipping": "Standard shipping takes 3-5 business days. Express delivery (1-2 days) is $15. Free shipping on orders over $500. International shipping to 45 countries.",
    "doc_returns": "Products can be returned within 30 days of purchase in original condition. Digital downloads are non-refundable. Refunds processed within 5-7 business days.",
}

def extract_propositions(text: str) -> list[str]:
    raw = [s.strip() for s in text.replace('\n', ' ').split('. ')]
    return [s + '.' if not s.endswith('.') else s for s in raw if len(s) > 15]

PDR_SCHEMA = {
    "index": {"name": "nb05_pdr", "prefix": "chunk5:", "storage_type": "json"},
    "fields": [
        {"name": "parent_id", "type": "tag"},
        {"name": "proposition", "type": "text"},
        {"name": "embedding", "type": "vector",
         "attrs": {"dims": 384, "algorithm": "flat", "distance_metric": "cosine"}}
    ]
}
pdr5 = SearchIndex(IndexSchema.from_dict(PDR_SCHEMA), redis_url=REDIS_URL)
pdr5.create(overwrite=True)

parent_store5 = {}
docs5 = []
for pid, txt in KNOWLEDGE_BASE.items():
    parent_store5[pid] = txt
    props = extract_propositions(txt)
    embs  = vectorizer.embed_many(props, as_buffer=False)
    for p, e in zip(props, embs):
        docs5.append({"parent_id": pid, "proposition": p, "embedding": e})

pdr5.load(docs5)
print(f"✅ PDR index ready ({len(docs5)} propositions)")

✅ PDR index ready (20 propositions)


In [7]:
# Cell 6: Run full evaluation against the golden dataset
import time

def evaluate_rag(golden: list, threshold: float = 0.40) -> list:
    """Evaluate RAG against golden dataset. Returns per-item scores."""
    results = []
    for item in golden:
        q_vec = vectorizer.embed(item["question"], as_buffer=False)
        vq = VectorQuery(q_vec, "embedding",
                         return_fields=["parent_id", "proposition"], num_results=1)
        retrieved = pdr5.query(vq)

        if not retrieved or float(retrieved[0]["vector_distance"]) > threshold:
            results.append({"question": item["question"],
                            "faithfulness": 0.0, "relevancy": 0.0,
                            "recall": 0.0, "retrieved_correct": False,
                            "status": "GUARD"})
            continue

        doc_id = retrieved[0]["parent_id"]
        context = parent_store5.get(doc_id, "")
        answer  = f"Based on our docs: {context[:200]}"

        results.append({
            "question":   item["question"],
            "faithfulness": faithfulness_score(answer, context),
            "relevancy":    answer_relevancy_score(item["question"], answer),
            "recall":       context_recall_score(item["ground_truth"], context),
            "retrieved_correct": doc_id == item["ideal_doc_id"],
            "status": "OK"
        })
    return results

eval_results = evaluate_rag(GOLDEN)

print(f"{'Q':50} {'Faith':6} {'Relev':6} {'Recall':6} {'Correct':7} {'Status'}")
print("-" * 95)
for r in eval_results:
    q_short = r['question'][:48]
    correct = '✅' if r['retrieved_correct'] else '❌'
    print(f"{q_short:50} {r['faithfulness']:6.3f} {r['relevancy']:6.3f} {r['recall']:6.3f} {correct:7} {r['status']}")

Q                                                  Faith  Relev  Recall Correct Status
-----------------------------------------------------------------------------------------------
What GPU does the Aether X1 have?                   0.909  0.760  0.600 ✅       OK
How long is the Sony XM5 battery life?              0.000  0.000  0.000 ❌       GUARD
What is the price of the Nebula Tab Pro?            0.900  0.750  0.667 ✅       OK
How long does standard shipping take?               0.900  0.746  1.000 ✅       OK
Can I return a product after 30 days?               0.882  0.609  0.667 ✅       OK
What does the Aether warranty cover?                0.909  0.765  0.500 ✅       OK
What storage options does the Nebula Tab Pro hav    0.900  0.736  0.667 ✅       OK
How much does express delivery cost?                0.900  0.726  0.500 ✅       OK


In [8]:
# Cell 7: Aggregate scores — overall RAG quality report
ok_results = [r for r in eval_results if r['status'] == 'OK']

avg_faith   = sum(r['faithfulness'] for r in ok_results) / len(ok_results) if ok_results else 0
avg_relev   = sum(r['relevancy']    for r in ok_results) / len(ok_results) if ok_results else 0
avg_recall  = sum(r['recall']       for r in ok_results) / len(ok_results) if ok_results else 0
correct_pct = sum(1 for r in ok_results if r['retrieved_correct']) / len(ok_results) * 100 if ok_results else 0

print("\n📊 RAG Evaluation Report")
print("=" * 40)
print(f"  Avg Faithfulness  : {avg_faith:.3f}  {'✅' if avg_faith > 0.5 else '⚠️ Low — hallucination risk'}")
print(f"  Avg Relevancy     : {avg_relev:.3f}  {'✅' if avg_relev > 0.6 else '⚠️ Low — off-topic answers'}")
print(f"  Avg Context Recall: {avg_recall:.3f} {'✅' if avg_recall > 0.5 else '⚠️ Low — missing facts'}")
print(f"  Correct Doc Hits  : {correct_pct:.0f}%  {'✅' if correct_pct >= 75 else '⚠️ Low — wrong documents returned'}")
print(f"  Guard Blocks      : {len(eval_results) - len(ok_results)} questions blocked")

if SHOW_INSIGHTS:
    print("\n💡 INSIGHT: faithfulness < 0.5 is a red flag.")
    print("   It means your answer contains words NOT found in the retrieved context.")
    print("   This is the definition of hallucination in a RAG system.")


📊 RAG Evaluation Report
  Avg Faithfulness  : 0.900  ✅
  Avg Relevancy     : 0.727  ✅
  Avg Context Recall: 0.657 ✅
  Correct Doc Hits  : 100%  ✅
  Guard Blocks      : 1 questions blocked


In [9]:
# Cell 8: Simulate hallucination detection + guard
HALLUCINATED_ANSWERS = [
    ("What GPU does Aether X1 have?",
     "The Aether X1 uses a solar-powered quantum 12-core processor.",
     parent_store5["doc_aether_x1"]),
    ("Battery life of Sony XM5?",
     "Sony XM5 lasts 120 hours and charges in 10 minutes.",
     parent_store5["doc_sony_xm5"]),
]

FAITHFULNESS_THRESHOLD = 0.40

print("🚨 Hallucination Guard Test")
print("-" * 60)
for question, bad_answer, context in HALLUCINATED_ANSWERS:
    score = faithfulness_score(bad_answer, context)
    blocked = score < FAITHFULNESS_THRESHOLD
    print(f"Q: {question}")
    print(f"   Answer      : {bad_answer[:70]}...")
    print(f"   Faithfulness: {score:.3f}  → {'🔴 BLOCKED' if blocked else '🟢 ALLOWED'}")
    print()

🚨 Hallucination Guard Test
------------------------------------------------------------
Q: What GPU does Aether X1 have?
   Answer      : The Aether X1 uses a solar-powered quantum 12-core processor....
   Faithfulness: 0.167  → 🔴 BLOCKED

Q: Battery life of Sony XM5?
   Answer      : Sony XM5 lasts 120 hours and charges in 10 minutes....
   Faithfulness: 0.400  → 🟢 ALLOWED



In [25]:
# Cell 9: Checkpoint
score = 0
h_score = faithfulness_score(
    "The Aether X1 uses solar quantum chips",
    parent_store5["doc_aether_x1"]
)
g_score = faithfulness_score(
    "RTX 4090 GPU with liquid cooling with 3-year global",
    parent_store5["doc_aether_x1"]
)

if h_score < 0.3:
    print(f"✅ Test 1 PASS: Hallucination faithfulness score is low ({h_score:.3f})")
    score += 1
else:
    print(f"❌ Test 1 FAIL: Hallucination score too high ({h_score:.3f}) — check scorer")

if g_score > 0.5:
    print(f"✅ Test 2 PASS: Grounded answer has high faithfulness ({g_score:.3f})")
    score += 1
else:
    print(f"❌ Test 2 FAIL: Grounded answer score too low ({g_score:.3f})")

if correct_pct >= 75:
    print(f"✅ Test 3 PASS: PDR retrieved correct document {correct_pct:.0f}% of the time")
    score += 1
else:
    print(f"❌ Test 3 FAIL: Correct retrieval rate {correct_pct:.0f}% is below 75%")

print(f"\n🏆 Score: {score}/3")

✅ Test 1 PASS: Hallucination faithfulness score is low (0.200)
✅ Test 2 PASS: Grounded answer has high faithfulness (0.600)
✅ Test 3 PASS: PDR retrieved correct document 100% of the time

🏆 Score: 3/3


In [18]:
parent_store5["doc_aether_x1"]

'The Aether X1 Flagship laptop features an NVIDIA RTX 4090 GPU with a proprietary liquid-cooling system. It has a 4K OLED 120Hz ProMotion display, 64GB DDR5 RAM, and 2TB NVMe SSD. The price is $2,499 with a 3-year global warranty.'

In [26]:
faithfulness_score(
    "RTX 4090 GPU with liquid cooling with 3-year global",
    parent_store5["doc_aether_x1"]
)

0.6

---
## 📝 Assignment: Build a RAG Regression Test Suite

**Task 1**: Implement `RAGEvaluator` — a class that runs full evaluation and returns a DataFrame-style summary.

**Task 2**: Implement `compare_versions()` — detect metric regressions between two RAG configs.

**Task 3 (Bonus)**: Find the optimal `distance_threshold` by running evaluation at [0.2, 0.3, 0.4, 0.5] and maximizing `faithfulness × recall`.


In [9]:
# ── ASSIGNMENT ────────────────────────────────────────────────────────────────
from redisvl.query import VectorQuery

# Cell 1: Golden dataset — 8 question/answer/context triples
GOLDEN = [
    {
        "question": "What GPU does the Aether X1 have?",
        "ground_truth": "The Aether X1 has an NVIDIA RTX 4090 GPU with liquid cooling.",
        "ideal_doc_id": "doc_aether_x1"
    },
    {
        "question": "How long is the Sony XM5 battery life?",
        "ground_truth": "The Sony WH-1000XM5 battery lasts 30 hours with ANC on and 40 hours without.",
        "ideal_doc_id": "doc_sony_xm5"
    },
    {
        "question": "What is the price of the Nebula Tab Pro?",
        "ground_truth": "The Nebula Tab Pro starts at $899.",
        "ideal_doc_id": "doc_nebula_tab"
    },
    {
        "question": "How long does standard shipping take?",
        "ground_truth": "Standard shipping takes 3-5 business days.",
        "ideal_doc_id": "doc_shipping"
    },
    {
        "question": "Can I return a product after 30 days?",
        "ground_truth": "No. Products can only be returned within 30 days of purchase.",
        "ideal_doc_id": "doc_returns"
    },
    {
        "question": "What does the Aether warranty cover?",
        "ground_truth": "Aether warranty covers manufacturing defects for 3 years.",
        "ideal_doc_id": "doc_warranty"
    },
    {
        "question": "What storage options does the Nebula Tab Pro have?",
        "ground_truth": "The Nebula Tab Pro comes in 256GB, 512GB, and 1TB storage options.",
        "ideal_doc_id": "doc_nebula_tab"
    },
    {
        "question": "How much does express delivery cost?",
        "ground_truth": "Express delivery costs $15 for 1-2 day delivery.",
        "ideal_doc_id": "doc_shipping"
    },
]
print(f"✅ Golden dataset: {len(GOLDEN)} Q&A pairs")

KNOWLEDGE_BASE = {
    "doc_aether_x1": "The Aether X1 Flagship laptop features an NVIDIA RTX 4090 GPU with a proprietary liquid-cooling system. It has a 4K OLED 120Hz ProMotion display, 64GB DDR5 RAM, and 2TB NVMe SSD. The price is $2,499 with a 3-year global warranty.",
    "doc_sony_xm5": "The Sony WH-1000XM5 headphones feature industry-leading active noise cancellation with dual-chip processing. Battery life is 30 hours with ANC on, 40 hours without. Supports LDAC, AAC, and SBC codecs. Price is $349.",
    "doc_nebula_tab": "The Nebula Tab Pro is a 12-inch OLED tablet with an Apple M3 chip. It supports the Aether Pencil 2nd generation. Storage options: 256GB, 512GB, 1TB. Starting price $899.",
    "doc_warranty": "All Aether products come with a 3-year global warranty covering manufacturing defects. Accidental damage is covered under AetherCare+ for $99/year. Claims via any Aether Store or online portal.",
    "doc_shipping": "Standard shipping takes 3-5 business days. Express delivery (1-2 days) is $15. Free shipping on orders over $500. International shipping to 45 countries.",
    "doc_returns": "Products can be returned within 30 days of purchase in original condition. Digital downloads are non-refundable. Refunds processed within 5-7 business days.",
}

def faithfulness_score(answer: str, context: str) -> float:
    """
    Faithfulness = fraction of meaningful answer words found in context.
    A simple but effective proxy for hallucination detection.
    """
    stop_words = {"the", "a", "an", "is", "are", "was", "it", "in", "of",
                  "and", "or", "to", "for", "with", "on", "this", "that"}
    context_words = set(context.lower().split()) - stop_words
    answer_words = [w for w in answer.lower().split()
                    if len(w) > 3 and w not in stop_words]
    if not answer_words:
        return 0.0
    hits = sum(1 for w in answer_words if w in context_words)
    return round(hits / len(answer_words), 4)

def cosine_similarity(vec_a: list, vec_b: list) -> float:
    a, b = np.array(vec_a), np.array(vec_b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

def answer_relevancy_score(question: str, answer: str) -> float:
    """Higher = answer is more on-topic to the question."""
    q_vec = vectorizer.embed(question, as_buffer=False)
    a_vec = vectorizer.embed(answer, as_buffer=False)
    return round(cosine_similarity(q_vec, a_vec), 4)

def context_recall_score(ground_truth: str, retrieved_context: str) -> float:
    """
    Context Recall = fraction of ground_truth key facts found in retrieved context.
    """
    stop_words = {"the", "a", "an", "is", "are", "was", "it", "in", "of",
                  "and", "or", "to", "for", "with", "on"}
    gt_words = [w for w in ground_truth.lower().split()
                if len(w) > 3 and w not in stop_words]
    ctx_words = set(retrieved_context.lower().split())
    if not gt_words:
        return 0.0
    hits = sum(1 for w in gt_words if w in ctx_words)
    return round(hits / len(gt_words), 4)

def extract_propositions(text: str) -> list[str]:
    raw = [s.strip() for s in text.replace('\n', ' ').split('. ')]
    return [s + '.' if not s.endswith('.') else s for s in raw if len(s) > 15]

PDR_SCHEMA = {
    "index": {"name": "nb05_pdr", "prefix": "chunk5:", "storage_type": "json"},
    "fields": [
        {"name": "parent_id", "type": "tag"},
        {"name": "proposition", "type": "text"},
        {"name": "embedding", "type": "vector",
         "attrs": {"dims": 384, "algorithm": "flat", "distance_metric": "cosine"}}
    ]
}
pdr5 = SearchIndex(IndexSchema.from_dict(PDR_SCHEMA), redis_url=REDIS_URL)
pdr5.create(overwrite=True)

parent_store5 = {}
docs5 = []
for pid, txt in KNOWLEDGE_BASE.items():
    parent_store5[pid] = txt
    props = extract_propositions(txt)
    embs  = vectorizer.embed_many(props, as_buffer=False)
    for p, e in zip(props, embs):
        docs5.append({"parent_id": pid, "proposition": p, "embedding": e})

pdr5.load(docs5)
print(f"✅ PDR index ready ({len(docs5)} propositions)")

class RAGEvaluator:
    """
    A reusable evaluation framework for RAG pipelines.
    """

    def __init__(self, pdr_index, parent_store: dict, vectorizer,
                 faithfulness_threshold: float = 0.40,
                 guard_threshold: float = 0.40):
        # Store all parameters
        self.pdr_index = pdr_index
        self.parent_store = parent_store
        self.vectorizer = vectorizer
        self.faithfulness_threshold = faithfulness_threshold
        self.guard_threshold = guard_threshold

    def evaluate(self, golden_dataset: list) -> list:
        """
        Run full evaluation on golden_dataset.
        For each item, compute all 4 metrics.
        """
        ## ADD YOUR CODE HERE
        pass


    def summary(self, results: list) -> dict:
        """
        Compute aggregate metrics.
        """
        ok_results = [r for r in results if r['status'] != 'GUARD']
        total_questions = len(results)
        guard_blocks = len(results) - len(ok_results)

        avg_faith = sum(r['faithfulness'] for r in ok_results) / len(ok_results) if ok_results else 0.0
        avg_relev = sum(r['relevancy'] for r in ok_results) / len(ok_results) if ok_results else 0.0
        avg_recall = sum(r['recall'] for r in ok_results) / len(ok_results) if ok_results else 0.0
        correct_pct = sum(1 for r in ok_results if r['retrieved_correct']) / len(ok_results) * 100 if ok_results else 0.0

        return {
            "avg_faithfulness": avg_faith, 
            "avg_relevancy": avg_relev, 
            "avg_recall": avg_recall,
            "correct_retrieval_pct": correct_pct, 
            "guard_blocks": guard_blocks, 
            "total_questions": total_questions
        }

    def print_report(self, results: list):
        """
        Print a formatted table + summary.
        """
        print(f"{'Q':50} {'Faith':6} {'Relev':6} {'Recall':6} {'Correct':7} {'Status'}")
        print("-" * 95)
        for r in results:
            q_short = r['question'][:48]
            correct = '✅' if r['retrieved_correct'] else '❌'
            print(f"{q_short:50} {r['faithfulness']:6.3f} {r['relevancy']:6.3f} {r['recall']:6.3f} {correct:7} {r['status']}")
            
        summ = self.summary(results)
        print("\n📊 RAG Evaluation Summary")
        print("=" * 40)
        print(f"  Avg Faithfulness  : {summ['avg_faithfulness']:.3f}")
        print(f"  Avg Relevancy     : {summ['avg_relevancy']:.3f}")
        print(f"  Avg Context Recall: {summ['avg_recall']:.3f}")
        print(f"  Correct Doc Hits  : {summ['correct_retrieval_pct']:.0f}%")
        print(f"  Guard Blocks      : {summ['guard_blocks']} questions blocked out of {summ['total_questions']}")


def compare_versions(summary_v1: dict, summary_v2: dict, regression_threshold: float = 0.05):
    """
    Compare two evaluation summaries and alert on regressions.
    """
    print(f"\n{'Metric':25} | {'v1_score':8} | {'v2_score':8} | {'Delta':8} | {'Status'}")
    print("-" * 75)
    
    metrics = ["avg_faithfulness", "avg_relevancy", "avg_recall", "correct_retrieval_pct"]
    has_regression = False
    
    for m in metrics:
        ## ADD YOUR CODE HERE
        
            
        print(f"{m:25} | {v1:8.3f} | {v2:8.3f} | {delta:8.3f} | {status}")
        
    if has_regression:
        raise ValueError("Deployment Halted: Regressions detected in RAG metrics.")
    else:
        print("\n✅ All metrics are stable or improved. Safe to deploy.")


# Bonus: threshold sweep
def optimal_threshold_sweep(golden: list, thresholds: list) -> float:
    """
    Find the threshold that maximizes faithfulness × recall.
    """
    print(f"\n{'Threshold':9} | {'Faithfulness':12} | {'Recall':8} | {'Combined Score'}")
    print("-" * 55)
    
    best_th = 0.0
    best_score = -1.0
    
    for th in thresholds:
        ## ADD YOUR CODE HERE
        pass
            
    print(f"\n🏆 Optimal threshold found: {best_th:.2f} (Combined Score: {best_score:.3f})")
    return best_th

# ── EVALUATION RUNNER ───────────────────────────────────────────────────────
try:
    print("Testing Evaluator V1 (Strict Guard)...")
    eval_v1 = RAGEvaluator(pdr5, parent_store5, vectorizer, guard_threshold=0.2)
    res_v1 = eval_v1.evaluate(GOLDEN)
    sum_v1 = eval_v1.summary(res_v1)
    
    print("\nTesting Evaluator V2 (Relaxed Guard)...")
    eval_v2 = RAGEvaluator(pdr5, parent_store5, vectorizer, guard_threshold=0.45)
    res_v2 = eval_v2.evaluate(GOLDEN)
    eval_v2.print_report(res_v2)
    sum_v2 = eval_v2.summary(res_v2)
    
    print("\nComparing V1 to V2:")
    # INCREASED THRESHOLD: Tolerate up to a 10% variance to account for the relaxed guard
    compare_versions(sum_v1, sum_v2, regression_threshold=0.10)
    
    print("\nRunning Threshold Sweep:")
    optimal_threshold_sweep(GOLDEN, [0.1, 0.2, 0.3, 0.4, 0.5])
    
except Exception as e:
    print(f'❌ Test failed: {e}')

✅ Golden dataset: 8 Q&A pairs
✅ PDR index ready (20 propositions)
Testing Evaluator V1 (Strict Guard)...

Testing Evaluator V2 (Relaxed Guard)...
Q                                                  Faith  Relev  Recall Correct Status
-----------------------------------------------------------------------------------------------
What GPU does the Aether X1 have?                   0.909  0.760  0.600 ✅       PASS
How long is the Sony XM5 battery life?              0.000  0.000  0.000 ❌       GUARD
What is the price of the Nebula Tab Pro?            0.900  0.750  0.667 ✅       PASS
How long does standard shipping take?               0.900  0.746  1.000 ✅       PASS
Can I return a product after 30 days?               0.882  0.609  0.667 ✅       PASS
What does the Aether warranty cover?                0.909  0.765  0.500 ✅       PASS
What storage options does the Nebula Tab Pro hav    0.900  0.736  0.667 ✅       PASS
How much does express delivery cost?                0.900  0.726  0.500 ✅  